## 데이터 수집 자동화

### 다나와 상품정보 스크래핑

#### 자동 로그인

- 패스워드 숨기기
- setx 명령어로 시스템에 저장방법
- powershell

##### 1.setx 명령어로 시스템에 저장방법
```powershell
> setx DANAWA_ID "아이디"
성공: 지정한값을 저장했습니다.
```

```powershell
> setx DANAWA_PASSWORD "비밀번호"
성공: 지정한 값을 저장했습니다.
```

- sysdm.cpl로 시스템 정보에서 환경변수 user 리스트에서 오픈

##### 2. keyring 으로 원도우 자격증명 저장소
- setx와 달리 아예 오픈 안됨
- 패키지 설치

In [1]:
!pip install keyring

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import keyring
from getpass import getpass

In [3]:
username = input('다나와 아이디')
password = getpass('다나와 비밀번호')

# 아이디 저장
keyring.set_password('danawa.com', 'user_id', username)
# 패스워드
keyring.set_password('danawa.com', username, password)

print('아이디/패스워드 저장완료')

아이디/패스워드 저장완료


In [6]:
user_id = keyring.get_password('danawa.com', 'user_id')
user_id

'personar'

In [8]:
password = keyring.get_password('danawa.com', user_id)
password is not None

True

In [3]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [4]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.danawa.com')

wait.until(
    EC.presence_of_element_located((By.ID, 'danawa_footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="4b4f00a1f0be1a9415394924ca623b59", element="f.E5AAFCC8D87C7A7A4C345CC2433AF638.d.74BC6B930C1CF9D79039F90A1F11BAE5.e.81")>

In [16]:
# 로그인 링크 확인후 클릭
driver.find_element(By.LINK_TEXT, '로그인').click()

In [17]:
# 로그인하기

id_input = driver.find_element(By.CSS_SELECTOR, 'input[type="text"]')
id_input.send_keys(user_id)


pass_input = driver.find_element(By.CSS_SELECTOR, 'input[type="password"]')
pass_input.send_keys(password)

In [18]:
login_btn = driver.find_element(By.CLASS_NAME, 'btn_login')
login_btn.click()

#### 동적 페이지 스크래핑

- https://prod.danawa.com/list/?cate=11255834&15main_11_02


In [5]:
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="4b4f00a1f0be1a9415394924ca623b59", element="f.E5AAFCC8D87C7A7A4C345CC2433AF638.d.E0E3069F19F1A85A0CB49152F4839AF6.e.6645")>

- select 태그 페이지 내에 1개만 존재

In [6]:
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
     option.click()

wait.until(
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)    

<selenium.webdriver.remote.webelement.WebElement (session="4b4f00a1f0be1a9415394924ca623b59", element="f.E5AAFCC8D87C7A7A4C345CC2433AF638.d.E0E3069F19F1A85A0CB49152F4839AF6.e.6645")>

In [7]:
products =driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]')
len(products)

90

In [8]:
product = products[1]
product.text

'2\n영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060\n5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / AMD / 라이젠 5000시리즈 / 라이젠5 / 버미어 / DDR4 / 16GB / M.2 / 256GB / NVIDIA / 그래픽 메모리: 8GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / LED쿨러 / 용도: 게임용\n혜택 최저가\n1,125,100원\n[G마켓] 삼성카드\n/\n무이자 최대 24개월\n기획전\n성능도 가격도 GOOD! 조립PC 인기상품 추천\n21.03. 등록\n브랜드로그\n의견 66\n4.8\n리뷰수\n(230)\n관심\n대량구매\n2위\n16GB, M.2 256GB\n1,181,800\n원\n5몰\nVS상품비교에 추가\n16GB, M.2 512GB\n1,259,670\n원\n5몰\nVS상품비교에 추가\n1위\n32GB, M.2 1TB\n1,539,640\n원\n5몰\nVS상품비교에 추가'

In [9]:
## 1번 인덱스 값에서 제품명, 제품스펙, 2번 인덱스 값에서 가격
product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')

[<selenium.webdriver.remote.webelement.WebElement (session="4b4f00a1f0be1a9415394924ca623b59", element="f.E5AAFCC8D87C7A7A4C345CC2433AF638.d.E0E3069F19F1A85A0CB49152F4839AF6.e.6932")>,
 <selenium.webdriver.remote.webelement.WebElement (session="4b4f00a1f0be1a9415394924ca623b59", element="f.E5AAFCC8D87C7A7A4C345CC2433AF638.d.E0E3069F19F1A85A0CB49152F4839AF6.e.6933")>,
 <selenium.webdriver.remote.webelement.WebElement (session="4b4f00a1f0be1a9415394924ca623b59", element="f.E5AAFCC8D87C7A7A4C345CC2433AF638.d.E0E3069F19F1A85A0CB49152F4839AF6.e.6934")>]

In [10]:
computer_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[1]
computer_info.text

'2\n영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060\n5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / AMD / 라이젠 5000시리즈 / 라이젠5 / 버미어 / DDR4 / 16GB / M.2 / 256GB / NVIDIA / 그래픽 메모리: 8GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / LED쿨러 / 용도: 게임용\n혜택 최저가\n1,125,100원\n[G마켓] 삼성카드\n/\n무이자 최대 24개월\n기획전\n성능도 가격도 GOOD! 조립PC 인기상품 추천\n21.03. 등록\n브랜드로그\n의견 66\n4.8\n리뷰수\n(230)\n관심\n대량구매'

In [11]:
computer_info.find_element(By.CSS_SELECTOR, 'div a').text

'영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060'

In [12]:
# 컴퓨터 사양

specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')

' / '.join(spec.text for spec in specs)

'5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / AMD / 라이젠 5000시리즈 / 라이젠5 / 버미어 / DDR4 / 16GB / M.2 / 256GB / NVIDIA / 그래픽 메모리: 8GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / LED쿨러 / 용도: 게임용'

In [13]:
price_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[2]
price_info.text

'2위\n16GB, M.2 256GB\n1,181,800\n원\n5몰\nVS상품비교에 추가\n16GB, M.2 512GB\n1,259,670\n원\n5몰\nVS상품비교에 추가\n1위\n32GB, M.2 1TB\n1,539,640\n원\n5몰\nVS상품비교에 추가'

In [14]:
price_info.find_element(By.CSS_SELECTOR, 'li').text

'2위\n16GB, M.2 256GB\n1,181,800\n원\n5몰\nVS상품비교에 추가'

In [15]:
# 인덱스로 데이터 가져오면서 필요값만 필터링
# 원이 나오면 반복 종료
price_info.find_elements(By.CSS_SELECTOR, 'li> div span')[0].text

'2위'

In [16]:
spans =price_info.find_elements(By.CSS_SELECTOR, 'li> div span')

for span in spans:
    if ('위' in span.text ):continue
    if ('원' in span.text):break
    print(span.text)

16GB, M.2 256GB
1,181,800


- 각 부분별 동작확인 완료

##### 한페이지 90개 데이터 스크래핑

In [29]:
# 다나와 게시판 게이밍PC 페이지로 이동
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
   # 모든 요소가 화면에 완전 표시될때까지 대기
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)

# 게시물 개수 30개에서 90개로 늘리기
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
     option.click()

# 화면에 90개 데이터를 다 조회할때까지 대기
products = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
) 

# list를 담을 수 있는 배열 생성
danawa_prices = []

# 총 게시물 읽어오기
# len(products)
for index, product in enumerate(products):
    # print(index, product.text)
    items = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')
    computer_info = items[1]  # 컴퓨터 명, 사양 정보 들어있는 영역
    price_info = items[2]   # 가격정보 들어있는 영역

    # print(computer_info.text)
    ## Pandas DF에 컬럼으로 사용할 이름들 사용
    ## index, computer_name, computer_spec
    computer_name = computer_info.find_element(By.CSS_SELECTOR, 'div a').text
    specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
    computer_spec = ' / '.join(spec.text for spec in specs)

    # 가격
    spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span') 
    price_spec = ''
    price = ''

    for span in spans:
        if ('위' in span.text): continue  # 순위 제외
        if ('원' in span.text): break    # 첫번째 금액만 받고 종료

        if ('B' in span.text):
            price_spec = span.text
        else:
            price = span.text

    danawa_prices.append({
        'index': index,
        'computer_name': computer_name,
        'computer_spec': computer_spec,
        'price_spec' : price_spec,
        'price' : price
    })       
    

In [ ]:
## Pandas DF으로 변경

df_danawa = pd.DataFrame(danawa_prices)

In [32]:
df_danawa.tail()

,index,computer_name,computer_spec,price_spec,price
85,85,MSI MAG Codex R2 A5NVM I12 U5-225F RTX 5060 Ti...,225F / RTX 5060 Ti / (인텔) H810 / OS미포함 / 600W ...,"16GB, M.2 1TB","1,759,000"
86,86,STCOM STAI-V138,5600 / RTX 5060 / (AMD) A520 / OS미포함 / 600W / ...,"16GB, M.2 512GB","1,210,710"
87,87,영웅컴퓨터 영웅 울트라게이밍 010,7500F / RTX 5070 Ti / (AMD) B650 / OS미포함 / 800...,"32GB, M.2 1TB","3,409,610"
88,88,다나와표준PC 게임용 260114,9600X / RX 9060 / (AMD) B650 / OS미포함 / 850W / ...,"16GB, M.2 1TB","1,921,100"
89,89,포유컴퓨터 퍼포먼스PC 46 R5 7400F RTX5060,7400F / RTX 5060 / (AMD) B650 / OS미포함 / 600W /...,"16GB, M.2 1TB","1,611,840"


### 페이지 확인

In [35]:
#
buttons = driver.find_elements(By.CSS_SELECTOR, 'nav > nav div button')
len(buttons)

10

In [ ]:
for button in buttons:
    print(button.text)

1
2
3
4
5
6
7
8
9
10


In [38]:
button_nextpage = buttons[9]
button_nextpage.click()

In [ ]:
# 다 동작했을때 10페이지에 있는지 확인
for button in buttons:
    button.click()

    products = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
) 

### 

In [41]:
# 다나와 게시판 게이밍PC 페이지로 이동
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    # 모든 요소가 화면에 완전표시될때까지 대기
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)

# 게시물 개수 30개에서 90개로 늘리기
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

# list를 담을 수 있는 배열 생성
danawa_prices = []
buttons = driver.find_elements(By.CSS_SELECTOR, 'nav > nav div button')

# 10개 페이지 이동하면서 스크래핑
for button in buttons:
    button.click()
    page_num = int(button.text)

    products = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
    )

    # 한페이지 스크래핑 내용이 여기 
    # 총 게시물 읽어오기
    # len(products)
    for index, product in enumerate(products):
        # print(index, product.text)
        items = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')
        computer_info = items[1]  # 컴퓨터 명, 사양 정보 들어있는 영역
        price_info = items[2]   # 가격정보 들어있는 영역

        # print(computer_info.text)
        ## Pandas DF에 컬럼으로 사용할 이름들 사용
        ## index, computer_name, computer_spec
        computer_name = computer_info.find_element(By.CSS_SELECTOR, 'div a').text
        specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
        computer_spec = ' / '.join(spec.text for spec in specs)

        # 가격
        spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span') 
        price_spec = ''
        price = ''

        for span in spans:
            if ('위' in span.text): continue  # 순위 제외
            if ('원' in span.text): break    # 첫번째 금액만 받고 종료

            if ('B' in span.text):
                price_spec = span.text
            else:
                price = span.text

        danawa_prices.append({
            'index': ((page_num - 1) * 90) + index,
            'computer_name': computer_name,
            'computer_spec': computer_spec,
            'price_spec' : price_spec,
            'price' : price
        })  
    

In [49]:
df_total = pd.DataFrame(danawa_prices)

df_total.info()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   index          900 non-null    int64
 1   computer_name  900 non-null    str  
 2   computer_spec  900 non-null    str  
 3   price_spec     900 non-null    str  
 4   price          900 non-null    str  
dtypes: int64(1), str(4)
memory usage: 35.3 KB


In [ ]:
# 가격 비교예정으로 금액이 없음
df_total[df_total['price'].isna()]

,index,computer_name,computer_spec,price_spec,price
395,395,MSI PRO DP180 14 i5-14400,i5-14400 / UHD 730 / (인텔) H610 / 윈도우11 프로 / 인텔...,,NaN


In [ ]:
# 가격이 없느 행 삭제
df_total=df_total.dropna()

In [50]:
# Pandas 컬럼에 천단위 쉼표(,)는 제거후 형 변환
df_total['price'] = df_total['price'].str.replace(',','')

In [57]:
df_total['price'] = pd.to_numeric(df_total['price'])

In [58]:
df_total.info()

<class 'pandas.DataFrame'>
Index: 899 entries, 0 to 899
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   index          899 non-null    int64  
 1   computer_name  899 non-null    str    
 2   computer_spec  899 non-null    str    
 3   price_spec     899 non-null    str    
 4   price          899 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 42.1 KB


In [59]:
df_total.to_csv('../data/danawa_게이밍PC.csv', encoding='utf-8')